# 🏋️ Notebook 03 — Model Training (YOLOv8s)
## Human Intrusion Detection System — Person Detection

**GPU**: NVIDIA Quadro T2000 (4GB VRAM)  
**Model**: YOLOv8s (small) — optimized for 4GB VRAM  
**Task**: Detect ONLY persons (1 class), no vehicles  

### Training Plan
| Setting | Value | Reason |
|---|---|---|
| Model | YOLOv8s | 11M params, fits 4GB VRAM |
| Batch Size | 8 | Safe headroom for T2000 |
| FP16 (AMP) | ✅ True | ~2× speed, halves VRAM |
| Image Size | 640 | YOLO standard |
| Epochs | 80 | With early stopping |
| Dataset | COCO 2017 + MOT17 | 64K+ person images |

---

In [ ]:
import sys, os, time, warnings
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import yaml
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Fix: Robust project root — works regardless of where Jupyter was launched
def _find_root():
# Fix: Robust root detection (works from any Jupyter launch directory)
def _find_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'src').is_dir() and (p / 'requirements.txt').exists():
            return p
    return Path.cwd()
ROOT = _find_root()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')
print(f'src/ found  : {(ROOT / "src").is_dir()}')
        if (p / "src").is_dir() and (p / "requirements.txt").exists():
            return p
    return Path.cwd()
ROOT = _find_root()
print(f"Project root: {ROOT} | src exists: {(ROOT/chr(39)+"src"+chr(39)).is_dir()}")
PROC_DIR = ROOT / 'data' / 'processed'
WEIGHTS  = ROOT / 'weights'
WEIGHTS.mkdir(exist_ok=True)

# ── GPU Status ────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'✅ GPU : {gpu.name}')
    print(f'   VRAM: {gpu.total_memory / 1e9:.2f} GB')
    print(f'   CUDA: {torch.version.cuda}')
    DEVICE = 'cuda'
else:
    print('⚠️  CUDA not available — using CPU (training will be very slow)')
    DEVICE = 'cpu'

print(f'\nProject root: {ROOT}')
print(f'Dataset dir : {PROC_DIR}')

IndentationError: unindent does not match any outer indentation level (<string>, line 30)

In [ ]:
# ── Verify dataset is ready ────────────────────────────────────
yaml_path = PROC_DIR / 'intrusion.yaml'

if not yaml_path.exists():
    raise FileNotFoundError(
        f'Dataset YAML not found: {yaml_path}\n'
        'Please run Notebook 02 — Preprocessing first!'
    )

with open(yaml_path) as f:
    ds_cfg = yaml.safe_load(f)

print('📄 Dataset Configuration:')
print(yaml.dump(ds_cfg, default_flow_style=False))

train_imgs = list((PROC_DIR / 'images' / 'train').glob('*.jpg'))
val_imgs   = list((PROC_DIR / 'images' / 'val').glob('*.jpg'))

print(f'Training images  : {len(train_imgs):,}')
print(f'Validation images: {len(val_imgs):,}')

if len(train_imgs) == 0:
    print('\n⚠️  No training images found! Run Notebook 02 first.')
else:
    print('\n✅ Dataset ready for training!')

In [ ]:
# ── Clear VRAM before training ─────────────────────────────────
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    used_mb  = torch.cuda.memory_allocated() / 1e6
    total_mb = torch.cuda.get_device_properties(0).total_memory / 1e6
    print(f'GPU Memory: {used_mb:.0f}MB used / {total_mb:.0f}MB total')
    print(f'Available : {total_mb - used_mb:.0f}MB free')
    print('✅ VRAM cleared for training.')

## 📥 Section 1 — Load Pretrained YOLOv8s

In [ ]:
# ── Download and inspect YOLOv8s pretrained on COCO ───────────
print('Loading YOLOv8s pretrained weights (COCO 80-class)...')
print('(Auto-downloads ~22MB on first run)')

model = YOLO('yolov8s.pt')

# Model info
print('\n📊 YOLOv8s Model Info:')
info = model.info(verbose=False)

# Count parameters
total_params = sum(p.numel() for p in model.model.parameters())
trainable    = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable:,}')

# VRAM estimate for inference
model_mb = total_params * 2 / (1024**2)  # FP16
print(f'  Model VRAM (FP16)   : {model_mb:.1f} MB')

print()
print('⚙️  Transfer Learning Strategy:')
print('  - Backbone: FROZEN for first 10 epochs (feature extraction)')
print('  - Head: UNFROZEN from start (person detection head)')
print('  - After epoch 10: Unfreeze all layers (fine-tuning)')
print('  - This prevents catastrophic forgetting on 4GB GPU')

## 🚀 Section 2 — Training

In [ ]:
# ── Training — Quadro T2000 Optimized ─────────────────────────
#
# IMPORTANT for 4GB VRAM:
#   • half=True    → FP16 training (AMP)
#   • batch=8      → safe batch size
#   • cache=False  → don't cache images in RAM (save memory)
#   • workers=4    → don't overload system RAM
#   • close_mosaic → disable mosaic last 10 epochs for stability

DATASET_YAML = str(PROC_DIR / 'intrusion.yaml')

print('🚀 Starting YOLOv8s Training...')
print(f'   Dataset: {DATASET_YAML}')
print(f'   Device : {DEVICE}')
print(f'   Epochs : 80 (with early stopping patience=20)')
print()

# Check dataset has images before starting
if len(train_imgs) == 0:
    print('⚠️  DEMO MODE: No training images. Showing training command only.')
    print()
    print('  Command to run:')
    print('  model.train(')
    print(f'      data="{DATASET_YAML}",')
    print('      epochs=80,')
    print('      batch=8,')
    print('      imgsz=640,')
    print('      device="cuda",')
    print('      half=True,       # FP16 — essential for 4GB VRAM')
    print('      cache=False,     # do not cache to RAM')
    print('      workers=4,')
    print('      patience=20,')
    print('      ...)')
else:
    # ── ACTUAL TRAINING ─────────────────────────────────────
    model = YOLO('yolov8s.pt')
    
    results = model.train(
        # ── Dataset ────────────────────────────
        data      = DATASET_YAML,
        
        # ── Training ───────────────────────────
        epochs    = 80,
        patience  = 20,          # early stopping
        batch     = 8,           # T2000: 4GB VRAM safe
        imgsz     = 640,
        
        # ── Device & Precision ─────────────────
        device    = DEVICE,
        half      = True,        # ✅ FP16 AMP — essential for 4GB GPU
        
        # ── Optimizer ──────────────────────────
        optimizer = 'AdamW',
        lr0       = 0.001,
        lrf       = 0.01,
        momentum  = 0.937,
        weight_decay = 0.0005,
        warmup_epochs = 3,
        
        # ── Augmentation ───────────────────────
        hsv_h     = 0.015,
        hsv_s     = 0.7,
        hsv_v     = 0.4,
        fliplr    = 0.5,
        scale     = 0.5,
        mosaic    = 1.0,
        mixup     = 0.1,
        copy_paste= 0.2,
        close_mosaic = 10,       # disable mosaic last 10 epochs
        
        # ── Memory & Speed ─────────────────────
        cache     = False,       # Don't cache — preserve VRAM
        workers   = 4,
        amp       = True,        # Automatic Mixed Precision
        
        # ── Logging ────────────────────────────
        project   = str(ROOT / 'runs' / 'train'),
        name      = 'intrusion_yolov8s_person',
        save      = True,
        save_period = 10,
        verbose   = True,
    )
    
    print('\n✅ Training Complete!')
    print(f'Best weights: {results.save_dir}/weights/best.pt')

## 📈 Section 3 — Monitor Training (Live & Post-Training)

In [ ]:
# ── Monitor GPU during training (run in separate cell) ─────────
# Run this cell WHILE training is happening in the cell above
# Or use: watch -n 1 nvidia-smi  in terminal

def print_gpu_stats():
    if not torch.cuda.is_available():
        return
    
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f'GPU Memory:')
    print(f'  Allocated : {allocated:.2f} GB')
    print(f'  Reserved  : {reserved:.2f} GB')
    print(f'  Total     : {total:.2f} GB')
    print(f'  Free      : {total - reserved:.2f} GB')
    
    # Bar chart
    usage_pct = (reserved / total) * 100
    bar_len = 40
    filled = int(bar_len * reserved / total)
    bar = '█' * filled + '░' * (bar_len - filled)
    color = '🟢' if usage_pct < 70 else ('🟡' if usage_pct < 90 else '🔴')
    print(f'\n  {color} [{bar}] {usage_pct:.1f}%')

print_gpu_stats()

In [ ]:
# ── Plot training curves from results.csv ─────────────────────
import pandas as pd

# Find the latest training run
runs_dir = ROOT / 'runs' / 'train'
run_dirs = sorted(runs_dir.glob('intrusion_yolov8s*')) if runs_dir.exists() else []

if run_dirs:
    latest_run = run_dirs[-1]
    csv_file = latest_run / 'results.csv'
    
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'YOLOv8s Training Curves — {latest_run.name}', 
                     fontsize=14, fontweight='bold')
        
        metrics_to_plot = [
            ('train/box_loss',     'Box Loss (Train)',    '#E74C3C'),
            ('train/cls_loss',     'Class Loss (Train)',  '#3498DB'),
            ('val/box_loss',       'Box Loss (Val)',      '#E67E22'),
            ('metrics/precision(B)','Precision',          '#2ECC71'),
            ('metrics/recall(B)',   'Recall',             '#9B59B6'),
            ('metrics/mAP50(B)',    'mAP@0.5 ← KEY',      '#1ABC9C'),
        ]
        
        for ax, (col, title, color) in zip(axes.flat, metrics_to_plot):
            if col in df.columns:
                epochs = df.iloc[:, 0]
                values = df[col]
                ax.plot(epochs, values, color=color, lw=2.5)
                ax.fill_between(epochs, values.rolling(3).min(), 
                               values.rolling(3).max(), alpha=0.15, color=color)
                
                # Mark best epoch
                if 'mAP' in col or 'precision' in col or 'recall' in col:
                    best_idx = values.idxmax()
                    ax.axvline(x=epochs[best_idx], color='gold', ls='--', lw=1.5)
                    ax.scatter(epochs[best_idx], values[best_idx], 
                              color='gold', zorder=5, s=80,
                              label=f'Best: {values[best_idx]:.4f}')
                    ax.legend(fontsize=9)
                
                ax.set_xlabel('Epoch', fontsize=10)
                ax.set_title(title, fontsize=11, fontweight='bold')
                ax.grid(True, alpha=0.3)
            else:
                ax.text(0.5, 0.5, f'No data:\n{col}', ha='center', va='center')
                ax.axis('off')
        
        plt.tight_layout()
        plt.savefig('training_curves.png', dpi=130, bbox_inches='tight')
        plt.show()
        
        # Summary
        print('\n📊 Training Summary:')
        final = df.iloc[-1]
        if 'metrics/mAP50(B)' in df.columns:
            best_map = df['metrics/mAP50(B)'].max()
            best_ep  = df['metrics/mAP50(B)'].idxmax()
            print(f'  Best mAP@0.5 : {best_map:.4f} (epoch {best_ep})')
        if 'metrics/precision(B)' in df.columns:
            print(f'  Best Precision: {df["metrics/precision(B)"].max():.4f}')
        if 'metrics/recall(B)' in df.columns:
            print(f'  Best Recall   : {df["metrics/recall(B)"].max():.4f}')
    else:
        print('Training in progress or not yet started...')
        print('Run training cell above first.')
else:
    print('No training runs found yet.')
    print('Complete training first, then re-run this cell.')

## 💾 Section 4 — Export & Save Best Weights

In [ ]:
# ── Export best weights to project weights/ folder ────────────
import shutil

runs_dir = ROOT / 'runs' / 'train'
run_dirs = sorted(runs_dir.glob('intrusion_yolov8s*')) if runs_dir.exists() else []

if run_dirs:
    latest_run = run_dirs[-1]
    best_pt = latest_run / 'weights' / 'best.pt'
    
    if best_pt.exists():
        # Copy to project weights dir
        dest = WEIGHTS / 'yolov8s_intrusion.pt'
        shutil.copy2(best_pt, dest)
        print(f'✅ Best weights saved: {dest}')
        print(f'   File size: {dest.stat().st_size / 1e6:.1f} MB')
        
        # ── Export to ONNX ───────────────────────────────────
        print('\nExporting to ONNX (for optimized inference)...')
        best_model = YOLO(str(dest))
        onnx_path = best_model.export(
            format   = 'onnx',
            imgsz    = 640,
            half     = True,       # FP16 ONNX
            simplify = True,       # simplify graph
            opset    = 17,
            dynamic  = False,      # fixed batch for deployment
        )
        print(f'✅ ONNX exported: {onnx_path}')
        
        # Copy ONNX to weights dir
        onnx_dest = WEIGHTS / 'yolov8s_intrusion.onnx'
        shutil.copy2(onnx_path, onnx_dest)
        print(f'✅ ONNX saved to: {onnx_dest}')
    else:
        print('⚠️  No best.pt found. Complete training first.')
else:
    print('No training runs found. Train the model first.')

In [ ]:
# ── Quick inference test on validation image ──────────────────
best_weights = WEIGHTS / 'yolov8s_intrusion.pt'

if best_weights.exists():
    model_trained = YOLO(str(best_weights))
    
    # Pick a random val image
    val_imgs = list((PROC_DIR / 'images' / 'val').glob('*.jpg'))
    if val_imgs:
        import random
        test_img = random.choice(val_imgs)
        
        # Inference
        results = model_trained.predict(
            source    = str(test_img),
            conf      = 0.5,
            iou       = 0.45,
            device    = DEVICE,
            verbose   = False,
        )
        
        r = results[0]
        img_bgr = cv2.imread(str(test_img))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle('Inference Test — Trained YOLOv8s Person Detector', 
                     fontsize=13, fontweight='bold')
        
        # Original
        axes[0].imshow(img_rgb)
        axes[0].set_title('Input Frame', fontsize=11)
        axes[0].axis('off')
        
        # With detections
        axes[1].imshow(img_rgb)
        if r.boxes is not None and len(r.boxes) > 0:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf = float(box.conf.item())
                rect = patches.Rectangle(
                    (x1, y1), x2-x1, y2-y1,
                    linewidth=2.5, edgecolor='#00FF88', facecolor='none'
                )
                axes[1].add_patch(rect)
                axes[1].text(x1, y1-6, f'person {conf:.2f}', fontsize=8,
                            color='#00FF88', fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.2',
                                      facecolor='black', alpha=0.6))
        
        n_det = len(r.boxes) if r.boxes is not None else 0
        axes[1].set_title(f'Detected: {n_det} person(s) | '
                          f'Inference: {r.speed["inference"]:.1f}ms', fontsize=11)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig('training_inference_test.png', dpi=130, bbox_inches='tight')
        plt.show()
        print(f'\n✅ Inference test complete')
        print(f'   Persons detected : {n_det}')
        print(f'   Inference time   : {r.speed["inference"]:.1f}ms')
        print(f'   FPS estimate     : {1000/r.speed["inference"]:.1f} FPS')
    else:
        print('No validation images found for test.')
else:
    print('Best weights not found. Complete training first.')

In [ ]:
print('='*60)
print('  ✅ Training Notebook Complete')
print('='*60)
print()
print('  What was done:')
print('  ├─ Loaded YOLOv8s pretrained on COCO')
print('  ├─ Fine-tuned on person-only intrusion dataset')
print('  ├─ GPU: Quadro T2000 | FP16 | batch=8')
print('  ├─ Exported best.pt to weights/')
print('  └─ Exported ONNX for deployment')
print()
print('  Files produced:')
for f in (ROOT / 'weights').glob('*'):
    size_mb = f.stat().st_size / 1e6
    print(f'  ├─ {f.name} ({size_mb:.1f} MB)')
print()
print('  ➡️  Next: Run Notebook 04 — Evaluation & Metrics')